In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd /content/drive/MyDrive/tinyllama_qlora

!pwd

!ls


/content/drive/MyDrive/tinyllama_qlora
/content/drive/MyDrive/tinyllama_qlora
notebooks  requirements.txt  src


In [3]:
# !pip uninstall -y -r requirements.txt

!pip install -r requirements.txt

In [4]:
import transformers
import datasets
import peft
import trl
import accelerate
import bitsandbytes

print("Transformers :", transformers.__version__)
print("Datasets     :", datasets.__version__)
print("PEFT         :", peft.__version__)
print("TRL          :", trl.__version__)
print("Accelerate   :", accelerate.__version__)
print("BitsAndBytes :", bitsandbytes.__version__)

Transformers : 4.51.0
Datasets     : 3.6.0
PEFT         : 0.16.0
TRL          : 0.19.1
Accelerate   : 1.8.1
BitsAndBytes : 0.46.1


In [5]:
import torch

print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA Available: True
GPU: Tesla T4


In [ ]:
from src.data.loader import load_data

DATASET_NAME = "databricks/databricks-dolly-15k"

dataset = load_data(DATASET_NAME)

print(dataset[0])

In [ ]:
from src.data.to_chat_format import format_chat

new_data_set = dataset.map(format_chat)

print(new_data_set[0])

In [ ]:
from src.config.tokenizer_loader import load_tokenizer
from src.data.apply_template import chat_template

MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = load_tokenizer(MODEL_NAME)

templated_data = new_data_set.map(
    lambda x: chat_template(x, tokenizer)
)

print(templated_data[0])

In [ ]:
from src.data.filter_data import remove

train_data = remove(templated_data)

print(train_data[0])

*LOAD LORA*

In [5]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

In [6]:
from src.config.bits_bytes_config import bit_byte_config

bnb_config = bit_byte_config()

print(bnb_config)

BitsAndBytesConfig {
  "_load_in_4bit": true,
  "_load_in_8bit": false,
  "bnb_4bit_compute_dtype": "float16",
  "bnb_4bit_quant_storage": "uint8",
  "bnb_4bit_quant_type": "fp4",
  "bnb_4bit_use_double_quant": true,
  "llm_int8_enable_fp32_cpu_offload": false,
  "llm_int8_has_fp16_weight": false,
  "llm_int8_skip_modules": null,
  "llm_int8_threshold": 6.0,
  "load_in_4bit": true,
  "load_in_8bit": false,
  "quant_method": "bitsandbytes"
}



In [8]:
from src.model.load_model import load_model

model = load_model(MODEL_NAME, bnb_config)

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:212: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [9]:
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

In [10]:
from src.config.lora_config import config_lora

lora_config = config_lora()

In [ ]:
from src.model.attach_lora import attach_lora

model = attach_lora(model, lora_config)

AttributeError: 'str' object has no attribute '__dict__'

In [ ]:
model.print_trainable_parameters()